# 05 · The `observation` table: combining definitions

`observation` holds facts that are not measurements, conditions, drugs or devices:
smoking, family history, past medical history. Each row pairs an `observation_concept_id`
(*what* was observed) with a value (a concept, a number or a text).

This notebook runs **three predefined definitions together**: pass a list of names,
YAML paths or both, and each note is sent to the LLM only once.

| Definition | What it extracts | Needs the mapper |
|---|---|---|
| `smoking_status` | status and pack-years | no |
| `family_history` | a relative's condition | yes, codes the condition |
| `medical_history` | the patient's past, resolved conditions | yes, codes the condition |

Details: [the observation table](https://precisiondata.github.io/aatm/getting-started/omop_tables/#the-observation-table).

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Mapper and notes

The same mapper as in notebook 04 codes the extracted names.

In [2]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline
from aatm.selectors import OpenAILLMSelector
from aatm.terminology_mapper import TerminologyMapper

mapper = TerminologyMapper(
    retriever="text-embedding-3-small", selector=OpenAILLMSelector("gpt-5-mini")
)
mapper.retriever.top_k = 30

notes = [
    ClinicalNote(
        text="Former smoker, 30 pack-years. Mother had breast cancer at 45. History of appendicitis.",
        person_id=20,
        note_date="2025-06-01",
    )
]

## 2. Run

Entity keys must be unique across the combined definitions; a clash fails with a message naming the files.

In [3]:
pipeline = OmopExtractionPipeline(
    ["smoking_status", "family_history", "medical_history"],
    model_id="openai/gpt-5-mini",
    mapper=mapper,
)
tables = await pipeline(notes)

tables["observation"][
    [
        "observation_source_value",
        "observation_concept_id",
        "value_as_concept_id",
        "value_as_number",
        "value_source_value",
        "qualifier_concept_id",
        "qualifier_source_value",
    ]
]

,observation_source_value,observation_concept_id,value_as_concept_id,value_as_number,value_source_value,qualifier_concept_id,qualifier_source_value
0,Former smoker,43054909,45883458,NaN,former,<NA>,NaN
1,30 pack-years,3003421,<NA>,30.0,30,<NA>,NaN
2,breast cancer,4167217,4112853,NaN,breast cancer,4248584,mother
3,appendicitis,3002660,440448,NaN,appendicitis,<NA>,NaN


How to read the rows:

- **Smoking** gives up to two rows: *Tobacco smoking status* with the status as
  `value_as_concept_id`, and *pack-years* with the number in `value_as_number`.
- **Family history** uses *Family history of clinical finding*, the condition in
  `value_as_concept_id` and the relative in `qualifier_concept_id` (a convention of this
  definition: the CDM has no relative column).
- **Medical history** uses LOINC *History of Past illness Narrative* with the condition in
  `value_as_concept_id`. Conditions the patient still has go to `condition_occurrence`
  (notebook 06).

## 3. Check the coded concepts and the issues

Only the conditions are coded by the mapper; smoking labels and relatives are fixed codes in the YAML.

In [4]:
tables.mappings_dataframe()[
    ["row_id", "column", "text", "concept_id", "concept_name", "domain_id", "score"]
]

,row_id,column,text,concept_id,concept_name,domain_id,score
0,3,value_as_concept_id,breast cancer,4112853,Malignant tumor of breast,Condition,0.791008
1,4,value_as_concept_id,appendicitis,440448,Appendicitis,Condition,0.839725


In [5]:
tables.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail


## Next

Diagnoses, signs and symptoms: [06_condition_occurrence.ipynb](06_condition_occurrence.ipynb).